# Pruebas de hipótesis: una cola y dos colas
**Autor:** Autoagente de Notebooks
**Fecha:** 2026-06-11
**Tags:** estadística, pruebas de hipótesis, una cola, dos colas, supervivencia
**Propósito:** Explicar teoría y proporcionar ejemplos reproducibles de pruebas de hipótesis según tipo de datos (continuas, proporciones, tasas, tiempo hasta evento).

## Objetivos
- Diferenciar pruebas de una cola y dos colas y cuándo usarlas.
- Presentar las pruebas estadísticas apropiadas según el tipo de datos y problema.
- Mostrar ejemplos prácticos en Python con datos sintéticos.

## Dos colas vs Una cola
- Prueba de dos colas: la hipótesis alternativa permite desviaciones en ambos sentidos (mayor o menor). Se usa cuando no se tiene dirección previa.
- Prueba de una cola: la hipótesis alternativa especifica una dirección (por ejemplo, mayor que). Se usa con justificación teórica o práctica.

Formalmente, para una media $\mu$:
- H0 (dos colas): $\mu = \mu_0$ vs H1: $\mu \neq \mu_0$.
- H0 (una cola, derecha): $\mu <= \mu_0$ vs H1: $\mu > \mu_0$.

El p-valor se interpreta acorde al tipo de prueba: para una cola se compara la probabilidad en la cola especificada; para dos colas se duplica la cola correspondiente cuando la estadística es simétrica.

## Mapeo de pruebas según tipo de datos
- Variables continuas (comparar medias):
  - Paramétricas (normal, varianzas iguales): prueba t de Student (independiente o pareada).
  - Varianzas desiguales: Welch t-test (`equal_var=False`).
  - No paramétricas: Mann–Whitney U (independientes), Wilcoxon signed-rank (pareadas).

- Proporciones o tablas de contingencia:
  - Chi-cuadrado de independencia (tabla grande).
  - Fisher exacto (muestras pequeñas o celdas con baja frecuencia).
  - Prueba de proporciones (z test) para comparar proporciones entre dos grupos.

- Tasas y conteos por tiempo (modelos de tasa):
  - Prueba de tasa de Poisson o test exacto de tasas (cuando el conteo por exposición se modela como Poisson).

- Tiempo hasta evento (supervivencia):
  - Log-rank test para comparar curvas de supervivencia entre grupos.
  - Modelos de Cox para ajustar covariables (no es una prueba simple, pero útil para inferencia).

## Ejemplos: imports y configuración

In [ ]:
# Imports y configuración
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

### Ejemplo 1: Comparación de medias (two-sample t-test y Welch)
Generamos dos grupos continuos y aplicamos t-test con y sin suposición de varianzas iguales, y Mann-Whitney si no son normales.

In [ ]:
# Datos sintéticos continuos
n1, n2 = 50, 45
mu1, mu2 = 0.5, 0.0
sigma1, sigma2 = 1.0, 1.5

g1 = np.random.normal(loc=mu1, scale=sigma1, size=n1)
g2 = np.random.normal(loc=mu2, scale=sigma2, size=n2)

# Student t-test (assume equal variance)
t_stat, p_val = stats.ttest_ind(g1, g2, equal_var=True)
print('Student t-test: t=%.3f, p=%.3f' % (t_stat, p_val))

# Welch's t-test (unequal variances)
t_stat_w, p_val_w = stats.ttest_ind(g1, g2, equal_var=False)
print("Welch t-test: t=%.3f, p=%.3f" % (t_stat_w, p_val_w))

# Mann-Whitney U (non-parametric)
u_stat, p_u = stats.mannwhitneyu(g1, g2, alternative='two-sided')
print('Mann-Whitney U: U=%.3f, p=%.3f' % (u_stat, p_u))

### Ejemplo 2: Proporciones (chi2 y z-test)
Contraste de proporciones entre dos grupos.

In [ ]:
# Tabla de contingencia ejemplo
# Grupo A: 30 éxitos de 200; Grupo B: 20 éxitos de 180
from scipy.stats import chi2_contingency

success = np.array([[30, 170], [20, 160]])  # [[success_A, fail_A],[success_B, fail_B]]
chi2, p, dof, exp = chi2_contingency(success)
print('Chi2 test: chi2=%.3f, p=%.3f' % (chi2, p))

# Prueba de proporciones (z-test) usando aproximación
try:
    from statsmodels.stats.proportion import proportions_ztest
    count = np.array([30, 20])
    nobs = np.array([200, 180])
    stat, pval = proportions_ztest(count, nobs)
    print('Proportions z-test: z=%.3f, p=%.3f' % (stat, pval))
except Exception as e:
    print('statsmodels no disponible: instala statsmodels para z-test (pip install statsmodels)')

### Ejemplo 3: Tiempo hasta evento (log-rank test)
Usaremos `lifelines` para realizar un log-rank entre dos grupos.

In [ ]:
# Log-rank test con lifelines (si está instalado)
try:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import logrank_test
    # Generar datos sintéticos de supervivencia exponencial
    n = 100
    # Grupo A con mayor tiempo medio
    t_A = np.random.exponential(scale=10, size=n)
    t_B = np.random.exponential(scale=7, size=n)
    # Censoring indicator (no censura simulada aquí, todos eventos observados)
    e_A = np.ones_like(t_A, dtype=int)
    e_B = np.ones_like(t_B, dtype=int)

    results = logrank_test(t_A, t_B, event_observed_A=e_A, event_observed_B=e_B)
    print('Log-rank test: p=%.4f' % results.p_value)
except Exception as exc:
    print('lifelines no instalado: pip install lifelines para ejecutar ejemplo de supervivencia')

## Conclusiones y recomendaciones
- Elige prueba de una cola sólo con justificación y a priori.
- Para medias: usa Welch si sospechas varianzas desiguales.
- Para proporciones: usa chi-cuadrado o Fisher según el tamaño de muestra.
- Para tiempos hasta evento: usa log-rank y modelos de Cox para ajuste.

## Reproducibilidad
Instala dependencias necesarias:

```bash
pip install numpy pandas scipy matplotlib statsmodels lifelines
```

Referencias: libros de estadística y documentación de `scipy`, `statsmodels`, `lifelines`.